In [ ]:
from pyannote.audio import Pipeline


pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-community-1", token='......')   

In [ ]:
import wave

import torch
import numpy as np

CHANNELS = 1
SAMPLE_RATE = 16000

def load_pcm_audio(file_path):
    with open(file_path, "rb") as f:
        pcm_bytes = f.read()

    # interpret as int16
    audio = np.frombuffer(pcm_bytes, dtype=np.int16)

    # convert to float32 and normalize to [-1, 1]
    waveform = torch.from_numpy(audio).float().unsqueeze(0)
    
    waveform = waveform / 32768.0

    
    return {
        "waveform": waveform,
        "sample_rate": SAMPLE_RATE
    }

In [ ]:
data = load_pcm_audio('/home/andrew/Documents/audio/server/.cache/2b23af74-2927-4d86-8920-c35477533a19-1.pcm')
data2 = load_pcm_audio('/home/andrew/Documents/audio/server/.cache/6ffd7481-8f6d-4430-858d-29f0d71c0a8a-1.pcm')

In [ ]:
wf = data["waveform"]

print("numel:", wf.numel())
print("min/max:", wf.min().item(), wf.max().item())
print("mean:", wf.mean().item())
print("std:", wf.std().item())
print("all zero:", torch.all(wf == 0).item())
print("has nan:", torch.isnan(wf).any().item())

In [ ]:
print(data['waveform'])

In [ ]:
output = pipeline([data, data2])

In [ ]:
for turn, speaker in output.speaker_diarization:
    print(f"Speaker {speaker} speaks from {turn.start:.1f}s to {turn.end:.1f}s")

print(output.speaker_embeddings.shape)
print(output.speaker_diarization.labels())
output.speaker_embeddings


In [ ]:
print(output.speaker_embeddings.shape)

In [ ]:
output2 = pipeline(load_pcm_audio('/home/andrew/Documents/audio/server/.cache/1.pcm'))

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

embeddings = np.concatenate([output.speaker_embeddings, output2.speaker_embeddings], axis=0)


pca = PCA(n_components=3)
embedding_2d = pca.fit_transform(embeddings)

# 3d scatter plot
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(embedding_2d[:2, 0], embedding_2d[:2, 1], embedding_2d[:2, 2], label='Audio 1')
ax.scatter(embedding_2d[2:, 0], embedding_2d[2:, 1], embedding_2d[2:, 2], label='Audio 2')
ax.legend()
ax.set_title('PCA of Speaker Embeddings')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
plt.show()

In [ ]:
# cos dist of embeddings 

from scipy.spatial.distance import cosine

embed00 = output.speaker_embeddings[0]
embed01 = output.speaker_embeddings[1]
embed10 = output2.speaker_embeddings[0]
embed11 = output2.speaker_embeddings[1]

distances = [cosine(embed00, embed10).item(), 
             cosine(embed01, embed10).item(),
             cosine(embed00, embed11).item(), 
             cosine(embed01, embed11).item(), 
             ]

mses = [
    np.mean((embed00 - embed10) ** 2).item(),
    np.mean((embed01 - embed10) ** 2).item(),
    np.mean((embed00 - embed11) ** 2).item(),
    np.mean((embed01 - embed11) ** 2).item(),
]

print("Cosine distances between embeddings:")
print(distances)
print("MSEs between embeddings:")
print(mses)